In [0]:
import pandas as pd
from pyspark.sql import SparkSession

In [0]:
df = pd.read_csv("https://raw.githubusercontent.com/Bhevendra/ML-Datasets/refs/heads/main/retail_data/sales_orders.csv")

In [0]:
spark = SparkSession.builder.getOrCreate()
df = spark.createDataFrame(df)


In [0]:
from pyspark.sql.functions import (
    col,
    from_json,
    explode,
    schema_of_json
)

# =========================
# STEP 1: Infer schema for ordered_products
# =========================

sample_ordered = (
    df.select("ordered_products")
    .filter(col("ordered_products").isNotNull())
    .first()[0]
)

ordered_schema = schema_of_json(sample_ordered)

# =========================
# STEP 2: Parse JSON column
# =========================

df_parsed = df.withColumn(
    "ordered_products_json",
    from_json(col("ordered_products"), ordered_schema)
)

# =========================
# STEP 3: Explode array of products
# =========================

df_exploded = df_parsed.withColumn(
    "product",
    explode(col("ordered_products_json"))
)

# =========================
# STEP 4: Infer schema for promotion_info
# =========================

promo_sample = (
    df_exploded
    .select("product.promotion_info")
    .filter(col("product.promotion_info").isNotNull())
    .first()[0]
)

promo_schema = schema_of_json(promo_sample)

# =========================
# STEP 5: Parse promotion_info JSON
# =========================

df_fixed = df_exploded.withColumn(
    "promo",
    from_json(col("product.promotion_info"), promo_schema)
)

# =========================
# STEP 6: Flatten final dataframe
# =========================

df_flat = df_fixed.select(
    "Customer_ID",
    "customer_name",
    "order_number",
    "order_datetime",
    "number_of_line_items",

    col("product.curr").alias("curr"),
    col("product.id").alias("product_id"),
    col("product.name").alias("product_name"),
    col("product.price").cast("int").alias("price"),
    col("product.qty").cast("int").alias("qty"),
    col("product.unit").alias("unit"),

    col("promo.promo_disc").alias("promo_disc"),
    col("promo.promo_id").alias("promo_id"),
    col("promo.promo_item").alias("promo_item"),
    col("promo.promo_qty").alias("promo_qty")
)

# Display final dataframe
display(df_flat)

In [0]:
df_flat.write.mode("overwrite").format("delta").saveAsTable("batch.raw_data.sales_orders")